# GPU Matrix Multiplication Optimization Tutorial

This notebook guides you through optimizing a matrix multiplication kernel from naive to **50+ TFLOPS**, progressively applying key GPU optimization techniques.

Special thanks: https://github.com/seb-v/fp32_sgemm_amd

Related Blog: https://seb-v.github.io/optimization/update/2025/01/20/Fast-GPU-Matrix-multiplication.html

### Environment Setup
Verify your ROCm/HIP environment:

In [9]:
!hipcc --version

HIP version: 7.2.26015-fc0010cf6a
AMD clang version 22.0.0git (https://github.com/RadeonOpenCompute/llvm-project roc-7.2.0 26014 7b800a19466229b8479a78de19143dc33c3ab9b5)
Target: x86_64-unknown-linux-gnu
Thread model: posix
InstalledDir: /opt/rocm-7.2.0/lib/llvm/bin
Configuration file: /opt/rocm-7.2.0/lib/llvm/bin/clang++.cfg


In [10]:
!rocminfo | grep -E 'Name|gfx'

  Name:                    AMD Ryzen 9 3900X 12-Core Processor
  Marketing Name:          AMD Ryzen 9 3900X 12-Core Processor
  Vendor Name:             CPU                                
  Name:                    gfx1100                            
  Marketing Name:          Radeon RX 7900 XTX                 
  Vendor Name:             AMD                                
      Name:                    amdgcn-amd-amdhsa--gfx1100         
      Name:                    amdgcn-amd-amdhsa--gfx11-generic   


---
## Problem: Matrix Multiplication C = A x B

For 4096x4096 FP32 matrices:
- **FLOPs**: 2 x 4096^3 = 137 trillion operations
- **Memory bandwidth needed** (naive): ~90.2 GB/s

![](https://seb-v.github.io/assets/images/graph1.jpg)

In terms of complexity, we have **computational complexity** `O(n^3)` and memory accesses `O(n^2)`. If we don’t think about architectural details, this is clearly a compute bound problem and our goal will be to be compute bound on the GPU.

## RDNA3 Memory Hierarchy
```
Global Memory (GMEM)    960 GB/s, high latency
         | load/store
LDS (Shared Mem)  ~29 TB/s, workgroup-shared (64KB per CU)
         | spill/fill
Registers     Per-thread, very fast (256 max per kernel)
```

![](https://seb-v.github.io/assets/images/graph2.jpg)

# Theoretical Performance

RDNA3 GPUs are made of arrays of WorkGroup Processors (WGP). Every WGP are split into 2 Compute Units (CUs), themself split into 2 SIMDs. A SIMD handles the work of multiple threads organized in waves (or warps for CUDA folks) and has a set of components to do some work (like arithmetic operations). For Floating point operations, there are two 32 way VALU units.


![](https://seb-v.github.io/assets/images/graph3.jpg)





##  Theoretical Performance Calculation

To determine the peak performance of the GPU, we use the following formula:


$$FLOPS = \text{Frequency} \times \text{Number of SIMDs} \times \text{FLOPs per SIMD}$$

### 1. Compute Capability per SIMD

In this architecture, every SIMD can issue **two** floating-point instructions per cycle (utilizing dual vALU units). Using **FMA (Fused Multiply-Add)** instructions, which count as two operations (one multiply, one add), the throughput is:

* **Calculation:** $32 \text{ (lane width)} \times 2 \text{ (instructions)} \times 2 \text{ (FMA ops)} = \mathbf{128}$ **FLOPs per cycle per SIMD**.

### 2. Total System Throughput

The 8060 S hardware configuration consists of **20 WGPs** (Work Group Processors), which equates to $20 \times 2 \times 2 = \mathbf{80}$ **total SIMDs**.

Assuming a GPU frequency of **2.5 GHz**:

* **Calculation:** $2.5 \times 10^9 \text{ Hz} \times 80 \text{ SIMDs} \times 128 \text{ FLOPs/cycle} = \mathbf{25.6 \text{ TFLOPS}}$.

---

##  Memory Bandwidth Analysis

When performing a **4096 × 4096** matrix multiplication ($C = A \times B$), we must account for the data movement overhead relative to the available system bandwidth.

### System Specs

* **Memory:** Dual-Channel DDR5-6800.
* **Available Bandwidth:** $\sim \mathbf{90 \text{ GB/s}}$.

### Bandwidth Overhead Calculation

For FP32 matrices ($4 \text{ bytes per element}$), a single matrix multiplication involves three matrices ($A, B, \text{ and } C$).

If the operation takes approximately $5.53 \text{ ms}$ ($5.53 \times 10^{-3} \text{ s}$), the required bandwidth is:

* **Data Volume:** $4096 \times 4096 \times 4 \text{ bytes} \times 3 \text{ matrices} \approx 201.3 \text{ MB}$.
* **Effective Bandwidth Required:** $\frac{4096 \times 4096 \times 4 \times 3}{5.53 \times 10^{-3}} = \mathbf{36.48 \text{ GB/s}}$.

> [!NOTE]
> With a required bandwidth of **36.48 GB/s** against a ceiling of **90 GB/s**, the system is well within its limits, though actual efficiency will depend on the kernel's ability to hide latency and reuse data in the LDS/Registers.



---
## Baseline Performance Results (RX 7900 XTX)
| Kernel | Time | GFLOPS | vs rocBLAS |
|--------|------|------|----------|
| 0 - rocBLAS | 4.5ms | 30,547 | 100% |
| 1 - Naive | 136ms | 1,010 | 3.3% |
| 2 - LDS tiling | 34ms | 4,018 | 13.1% |
| 3 - Register tile | 6ms | 22,777 | 74.6% |
| 4 - GMEM dbl buffer | 5.4ms | 25,560 | 83.7% |
| 5 - LDS optimization | 4.1ms | 33,527 | **109.8%** |
| 6 - Dual FMA (ISA) | 3.6ms | 37,791 | **123.7%** |
| 7 - Loop unroll | 3.3ms | 41,256 | **135.1%** |
| 8 - Batched GMEM | 2.8ms | 49,047 | **160.6%** |

## Compile command

In [ ]:
!cd ../fp32_sgemm_amd && bash ./build.sh

---

## Kernel 0: The "Gold Standard" (rocBLAS)

We start by running **rocBLAS**, AMD's highly optimized library. This is our "Target to Beat."

* **Performance:** ~305 GFLOPS.
* **Efficiency:** This represents a well-tuned baseline that uses every trick in the book.

In [16]:
!! cd ../fp32_sgemm_amd && ./sgemm -k 0

['Running: Kernel 0 : ROCBLAS',
 '4.23824 ms -> 32440.2 GFLOPS',
 '--------------------']

---
## Kernel 1: Naive Implementation
**Approach**: One thread per C[i,j] element.

```cpp
__global__ void kernel1_naive(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float acc = 0.0f;
        for (int k = 0; k < N; ++k)
            acc += A[row * K + k] * B[k * N + col];
        C[row * N + col] = acc;
    }
}
```

In [ ]:
! cd ../fp32_sgemm_amd && ./sgemm -k 1

### Performance & Reality Check

* **Result:** ~1,010 GFLOPS (Only 3.3% of rocBLAS).
* **The Problem:** Low **Arithmetic Intensity**. For every multiplication (2 FLOPs), we are doing multiple 4-byte memory reads. We are "Memory Bound."

**Interactive Challenge:** Look at the loop `acc += A[row * K + k] * B[k * N + col]`. If a workgroup has 256 threads, how many times is the same value of `A[row][k]` loaded from Global Memory?
*(Hint: Every thread in the same row needs it!)*

In [11]:
!rocprofv3 --stats  --kernel-trace  -d prof_result --output-format csv --  ../fp32_sgemm_amd/sgemm -k 1

W20260313 16:28:16.648148 138120615944576 simple_timer.cpp:55] [rocprofv3] tool initialization ::     0.000557 sec
W20260313 16:28:16.652638 138120615944576 simple_timer.cpp:55] [rocprofv3] '../fp32_sgemm_amd/sgemm -k 1' ::     0.000000 sec
Running: Kernel 1 : Naive
W20260313 16:28:16.719362 138120615944576 tool.cpp:2422] HSA version 8.20.0 initialized (instance=0)
W20260313 16:28:18.080713 138120615944576 tool.cpp:3104] [PPID=237287][PID=245410][TID=245410][rocprofv3_error_signal_handler] rocprofv3 caught signal 2...
W20260313 16:28:18.080788 138120615944576 tool.cpp:3127] [PPID=237287][PID=245410][TID=245410][rocprofv3_error_signal_handler] rocprofv3 will wait for 0 children to exit
W20260313 16:28:18.080796 138120615944576 tool.cpp:3142] [PPID=237287][PID=245410][TID=245410][rocprofv3_error_signal_handler] rocprofv3 finalizing after signal 2...
^C


In [15]:
!rocprofv3 --hip-trace --pmc  --kernel-trace --pc-sampling-beta-enabled --output-format pftrace  -d prof_result --  ../fp32_sgemm_amd/sgemm -k 1

W20260313 16:38:18.672125 134973960946048 simple_timer.cpp:55] [rocprofv3] tool initialization ::     0.000849 sec
W20260313 16:38:18.672182 134973960946048 tool.cpp:2422] HIP (compiler) version 7.2.0 initialized (instance=0)
W20260313 16:38:18.682632 134973960946048 simple_timer.cpp:55] [rocprofv3] '../fp32_sgemm_amd/sgemm -k 1' ::     0.000000 sec
Running: Kernel 1 : Naive
W20260313 16:38:18.721540 134973960946048 tool.cpp:2422] HIP (runtime) version 7.2.0 initialized (instance=0)
W20260313 16:38:18.749346 134973960946048 tool.cpp:2422] HSA version 8.20.0 initialized (instance=0)
194.236 ms -> 707.846 GFLOPS
--------------------
W20260313 16:38:22.102778 134973960946048 simple_timer.cpp:55] [rocprofv3] '../fp32_sgemm_amd/sgemm -k 1' ::     3.420145 sec
[847.779]       perfetto.cc:47304 Configured tracing session 1, #sources:1, duration:0 ms, #buffers:1, total buffer size:1048576 KB, total sessions:1, uid:0 session name: ""
E20260313 16:38:22.598939 134973960946048 output_stream.cpp:1

**Analysis**:
- Low arithmetic intensity: 2N FLOPs / (3 x N x 4 bytes) = 0.167 F/byte
- Non-coalesced memory access on B[]
- Same A[row] loaded redundantly by many threads

---
## Kernel 2: LDS Tiling
**Optimization**: Load tile into shared memory (LDS), reuse across wave.

![](https://seb-v.github.io/assets/images/graph6b.jpg)

The main issue with our naive kernel is that our inner loop directly accesses global memory. This is inefficient because fetching data from global memory has a high latency, typically on the order of hundreds of cycles. Since each memory read is followed by minimal computation (just one multiplication and one addition), the GPU struggles to hide this latency, even with a large number of concurrent threads. Moreover, the algorithm repeatedly reads the same rows and columns from global memory across different threads, leading to redundant memory accesses and further exacerbating the performance bottleneck.

A solution to this problem is to load the data once into faster local memory and then iterate efficiently over it with all the threads. On RDNA3, we have the Local Data Store (LDS, or Shared Memory), a high-speed, low-latency memory accessible by all threads within a workgroup.

```cpp
#define TILE_SIZE 32
__global__ void kernel2_lds(const float *A, const float *B, float *C, int N)
{
    __shared__ float As[TILE_SIZE][TILE_SIZE];
    __shared__ float Bs[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;

    float sum = 0.0f;

    for (int t = 0; t < N; t += TILE_SIZE)
    {
        Bs[threadIdx.y][threadIdx.x] = B[N * (threadIdx.y + t) + col];
        As[threadIdx.y][threadIdx.x] = A[N * row + t + threadIdx.x];

        __syncthreads();

        for (int k = 0; k < TILE_SIZE; k++)
        {
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        }

        __syncthreads();
    }

    if (row < N && col < N)
    {
        C[row * N + col] = sum;
    }
}
```

In [17]:
! cd ../fp32_sgemm_amd && ./sgemm -k 2

Running: Kernel 2 : LDS
34.3918 ms -> 3997.73 GFLOPS
--------------------


**The Fix:** Instead of every thread hitting Global Memory individually, the whole workgroup cooperatively loads a "tile" of data into the **LDS (Shared Memory)**.


**Benefits**:
- Reuses data: Each A[i,k] used 32 times (once per thread in row)
- Arithmetic intensity: ~5.3 F/byte = 32x improvement
- Coalesced loads: Neighboring threads load contiguous GMEM


### Background: Data Reuse

By loading a $32 \times 32$ tile into LDS, each piece of data is loaded **once** but used **32 times** by the threads in that workgroup.

* **Arithmetic Intensity:** Increases by ~32x.
* **Result:** ~4,018 GFLOPS.

---
## Kernel 3: Register Tiling
**Optimization**: Further tile in registers, compute 64 elements per thread (8x8).

![](https://seb-v.github.io/assets/images/graph10.jpg)

In [ ]:
! cd ../fp32_sgemm_amd && ./sgemm -k 3

```cpp

   #define BLOCK_SIZE 256
    // Block Tile size
    constexpr int BN = 128;
    constexpr int BM = 128;

    // Number of Row or column we read per batch
    constexpr int BK = 8; 

    // Thread Tile size . 4x4
    constexpr int TN = 4;
    constexpr int TM = 4;

    // A wave is a block of 8x4 of the output matrix
    constexpr int nbThreadXPerWave = 8;
    constexpr int nbThreadYPerWave = 4;

    // Number of waves in a block
    constexpr int nbWavesPerBlock = BLOCK_SIZE / 32;

    constexpr int WN = 64;
    constexpr int WM = BN * BM / nbWavesPerBlock / WN;

    constexpr int nbIterWaveN = WN / (nbThreadXPerWave * TN);
    constexpr int nbIterWaveM = WM / (nbThreadYPerWave * TM);

    // LDS Tile
    __shared__ float As[BK][BM];
    __shared__ float Bs[BK][BN];
    
    // Column and row from A and B, stored into registers
    float A_col[nbIterWaveM * TM];
    float B_row[nbIterWaveN * TN];

    //Wave Tile (registers)
    float C_regs[TM * nbIterWaveM * TN * nbIterWaveN] = {0.0f};

```


Key changes from Kernel 2:
- Compute **64 elements per thread** (BLOCK_SIZE_M x BLOCK_SIZE_N = 8x8)
- Accumulate partial sums in registers: `float r[8][8]`
- Only write to GMEM once at the end

**Benefits**:
- Less LDS traffic (no intermediate reads/writes)
- Higher ILP: More independent ops for out-of-order execution
- Better occupancy: Wave stays busy while waiting on memory

---
## Kernel 4: GMEM Double Buffering


Problem here: With our current implementation, every wave must wait for global memory and then LDS write latency before doing any work. In a high-occupancy scenario, this shouldn’t be an issue if the GPU can find other waves to hide this latency. However, in practice, we often have multiple waves in the same state running simultaneously because we use a sync thread before and after reading from global memory.


![](https://seb-v.github.io/assets/images/graph14.jpg)

![](https://seb-v.github.io/assets/images/graph15.jpg)

**Optimization**: Hide load latency with two LDS tiles in flight.

![](https://seb-v.github.io/assets/images/graph15b.jpg)



In [ ]:
! cd ../fp32_sgemm_amd && ./sgemm -k 4

```cpp
// Pre-load next tile while computing current
As[(t/BK+1)%2][threadIdx.y][threadIdx.x] = A[...];  // Load tile T+1
__syncthreads();
sum += As[t/BK%2][...] * Bs[...] ;                   // Compute with tile t
```

**Background: Latency Hiding.** Loading from Global Memory (GMEM) takes hundreds of cycles. While we wait for Tile $T+1$ to arrive, our ALU is sitting idle.

**The Fix:** Use two sets of LDS buffers.

1. Compute using **LDS Buffer A** (Tile $T$).
2. Simultaneously, use the "Wait Time" to fetch Tile $T+1$ into **LDS Buffer B**.

**Benefits**:
- Latency hiding: GMEM takes ~400 cycles; computation overlaps
- VALU utilization higher

---
## Kernel 5: LDS Optimization

Now we are getting into the "Pro" territory. Even LDS has bottlenecks.

### The Background: LDS Bank Conflicts

LDS is organized into 32 banks. If two threads try to access different data in the same bank at the same time, the hardware stalls.

![](https://seb-v.github.io/assets/images/graph20.jpg)

**Optimizations**:
1. **Padding**: `__shared__ float As[BK][BM+4]` - avoid bank conflicts
2. **CU mode**: `-mcumode` - use smaller workgroup size (64 threads)
3. **Larger wavefront tile**: 16x8 instead of 8x8

In [ ]:
! cd ../fp32_sgemm_amd && ./sgemm -k 5

**Bank conflicts**:
LDS has 32 banks, each 4 bytes. Without padding threads access same bank.
Solution: `As[BK][BM + PADDING_DIV_4]` to stagger accesses.

**Result**: 33,527 GFLOPS - first kernel to beat rocBLAS!

---
## Kernel 6: ISA-level Dual FMA
**Optimization**: Directly write AMD GCN assembly for precise control.



### 1. Why Rewrite in Assembly?

![](https://seb-v.github.io/assets/images/graph25.jpg)
![](https://seb-v.github.io/assets/images/graph26.jpg)

In the RDNA3 architecture, **Dual FMA** allows the GPU to execute two floating-point multiply-add instructions ($C = A \times B + C$) in a single clock cycle. However, this "secret move" has strict hardware requirements:

* **Independence**: The two instructions cannot depend on each other's results.
* **VGPR Bank Constraints**: VGPRs are divided into four "banks" (Bank 0–3).
* Each bank acts like a cache with dedicated read ports.
* **The Conflict**: If two instructions simultaneously attempt to access the same bank via the same port, the hardware stalls for 2–3 cycles.


* **Parity Rules**: One destination register must be even-numbered, and the other must be odd.

**The Problem**: While the HIP compiler can generate these instructions, its register allocation is often disorganized, leading to frequent bank conflicts that bottleneck the GPU.


### 2. The Core Optimization Logic

We abandoned the compiler’s default mapping and designed a **perfectly symmetrical** register access scheme.

#### A. "Strategic Segregation" of Banks

To eliminate conflicts, we assigned matrices to specific "territories":

* **Matrix B**: Restricted to Bank 0 and Bank 1.
* **Matrix A**: Restricted to Bank 2 and Bank 3.
* **Accumulator C**: Distributed evenly across Banks 0, 1, 2, and 3.
* **The Trade-off**: Because registers are no longer strictly contiguous, the 128-bit loads (`ds_load_b128`) had to be replaced with 64-bit loads (`ds_load_b64`), but the gain in computational efficiency far outweighed this cost.

#### B. The Symmetrical Access Pattern

We designed a consistent sequence (shown in Figure 34 of the tutorial) where the X and Y instructions in a Dual FMA pair cross-read different banks. This ensures maximum **VGPR Cache** hits and zero port contention.

#### C. The Workflow

1. **Export ISA**: Use `hipcc --save-temps` to generate the raw `.s` assembly file.
2. **Manual Overhaul**: Use a custom tool or manual editing to replace the inner loop's 128 FMA instructions with the optimized `v_dual_fmac` pattern.
3. **Restore Mapping**: Since register numbers were changed for the optimization, use `v_mov` instructions after the loop to move data back to the original registers so the global memory write-back still works.



In [ ]:
! cd ../fp32_sgemm_amd && ./sgemm -k 6

---

## Kernel 7 & 8

*These kernels have expertise optimise techniques r. I suggest reading here directly from https://seb-v.github.io/optimization/update/2025/01/20/Fast-GPU-Matrix-multiplication.html*

In [ ]:
! cd ../fp32_sgemm_amd && ./sgemm -k 7



##  Kernel 7: ISA-Level Loop Unrolling

**The Goal:** Eliminate loop overhead and maximize Instruction Level Parallelism (ILP) by taking full control of the Instruction Set Architecture (ISA).

### Key Techniques

* **Manual ISA Unrolling:** Unlike C++ compiler unrolling (which often leads to register pressure or excessive LDS pre-fetching), the assembly code for the core multiplication and load block was duplicated **8 times**.
* **Removal of Loop Logic:** The overhead of the loop was eliminated by removing:
* `s_cmpk_lg_i32`: The integer comparison used to check loop bounds.
* `s_cbranch_scc1`: The conditional branch instruction at the end of each iteration.


* **Manual Address Management:** Instead of relying on the compiler to track indices, addresses for matrices A and B are manually incremented using `v_add_nc_u32_e32` within the unrolled sequence.

### Results

* **Performance:** 41,255.6 GFLOPS (135.1% of rocBLAS).
* **Efficiency:** VALU (Vector ALU) utilization climbed above **80%**.
* **Bottleneck Identified:** High latency (up to 20 million clock cycles) caused by global memory (GMEM) load dependencies.




## Kernel 8: Batched GMEM Loads & SGPR Addressing

**The Goal:** Reduce the massive latency overhead of global memory loads by simplifying address calculations and reducing wave contention.

### Key Techniques

* **SGPR-Based Base Addressing:** The kernel was modified to use **SGPRs (Scalar General Purpose Registers)** for base addresses instead of computing unique 64-bit addresses in VGPRs for every single load.
* *Old Way:* 128 lines of code to compute 16 individual VGPR addresses.
* *New Way:* 16 lines using `global_load_b32 v_dest, v_offset, s[base_addr]`.


* **Pre-computed Offsets:** Matrix base addresses are loaded once into SGPRs at the start. Only a single **VGPR offset** is maintained and incremented, significantly reducing the VALU instructions needed for memory bookkeeping.
* **Instruction Scheduling (Chunking):** Instead of issuing all 16 global loads at once (which causes waves to compete for memory bandwidth at the barrier), the loads were split into **chunks of 2** and spread across the inner loop.
* This reduces the "Memory State" overlap between different wavefronts on the same SIMD.



### Results

* **Performance:** **49,047 GFLOPS** (~50 TFLOPS).
* **Efficiency:** **160.6%** of rocBLAS performance.
* **Latency Reduction:** Total latency for the loading process dropped from ~20 million cycles to **less than 2 million cycles**.



In [ ]:
! cd ../fp32_sgemm_amd && ./sgemm -k 8

---
## How to Run the Full Test Suite

In [ ]:
!cd ../fp32_sgemm_amd && ./build.sh

In [ ]:
! cd ../fp32_sgemm_amd && ./sgemm

---
## Profiling with rocprof
See what limits performance:

```bash
rocprofv3 --stats-db ./perf_stats.db ./sgemm
rocprof-parser stats.db | head -100
```

**Key metrics to watch**:
- GMEM throughput: Should approach 960 GB/s
- LDS utilization: Higher is better (80%+ ideal)
- VALU FMA32 Utilization: Target 75%+
- Wavefront stall reasons: Identify bottlenecks

---
## Summary: Optimization Journey

| Step | Technique | Key Insight |
|-----|---------|-------------|
| 1->2 | **LDS tiling** | Shared memory = fast cache for workgroup |
| 2->3 | Register tiling | Process multiple outputs per thread |
| 3->4 | **GMEM double buffer** | Overlap load and compute |
| 4->5 | **LDS bank fix + CU mode** | Avoid LDS stalls, better occupancy |
| 5->6 | **ISA dual FMA** | Precise register allocation |
| 6->7 | Loop unrolling | Schedule ops optimally |
| 7->8 | Batched loads | Reduce instruction overhead |

## References
- [Original Blog](https://seb-v.github.io/optimization/update/2025/01/20/Fast-GPU-Matrix-multiplication.html)
- [ROCm Documentation](https://rocm.docs.amd.com/)
- [RDNA3 ISA Guide](https://www.amd.com/en/developer/resources/rdna-architectures)